# Day 3 Lab - Write it yourself: model, training, grid search, metrics

In the code breaks you *changed* numbers. Now you **write the code yourself** - step by step:
- **Part 1 - Build it yourself** (about 30 min): a model from a plan, then the training loop
- **Part 2 - Grid search** (about 20 min): try many settings automatically, choose the best one fairly
- **Part 3 - The right metric in the right place** (about 25 min): EEG seizure detection on CHB-MIT

What is not finished in class is homework. A **CPU runtime is enough**.

How the writing tasks work:
1. read the **plan** (a small table),
2. write your code below the numbered steps - where it says `pass  # YOUR CODE HERE`,
3. stuck? open **💡 Hint**,
4. run the **check cell**: ✅ = correct, ❌ = it tells you what is wrong.

You cannot break anything - if something looks strange: *Runtime → Restart session*, then run the setup cells again.

## Setup - run these two cells first
The first cell loads FashionMNIST (28 × 28 pixel images of 10 kinds of clothes): 10 000 training,
2 000 validation and 2 000 test images. Each image is turned into one row of 784 numbers and standardised
with the mean and spread of the *training* images only.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, TensorDataset

data = torchvision.datasets.FashionMNIST(root="data", train=True, download=True)
X = data.data.float().view(len(data), -1) / 255     # 60 000 images, each 784 pixel values
y = data.targets

X_train, y_train = X[:10000], y[:10000]             # training images
X_val, y_val = X[50000:52000], y[50000:52000]       # validation images: for choosing
X_test, y_test = X[52000:54000], y[52000:54000]     # test images: only at the very end
mean, std = X_train.mean(), X_train.std()           # computed on the training images only
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std
print("training:", len(X_train), "  validation:", len(X_val), "  test:", len(X_test))

The second cell defines the **check cells** and other helpers for the whole lab.
**Just run it - you do not need to read it.** (It does *not* contain the training loop - that is your job.)
PyTorch has no built-in metrics, so we use scikit-learn **only for the metrics**.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, average_precision_score, precision_score, recall_score


def _layers(model):
    """[('Linear', 784, 256), ('ReLU',), ('Dropout', 0.2), ...] for an nn.Sequential."""
    out = []
    for layer in model:
        if isinstance(layer, nn.Linear):
            out.append(("Linear", layer.in_features, layer.out_features))
        elif isinstance(layer, nn.Dropout):
            out.append(("Dropout", layer.p))
        else:
            out.append((type(layer).__name__,))
    return out


def _compare(model, plan, n_params):
    """Compare a model with the plan; print ✅ or the first difference."""
    if model is None:
        print("❌ no model yet - write it in the cell above and run it again")
        return False
    if not isinstance(model, nn.Sequential):
        print("❌ please build the model with nn.Sequential(...)")
        return False
    mine = _layers(model)
    if len(mine) != len(plan):
        print("❌ the plan has", len(plan), "layers, your model has", len(mine), "-", [m[0] for m in mine])
        return False
    for i, (m, p) in enumerate(zip(mine, plan)):
        if m[0] != p[0]:
            print("❌ layer", i + 1, "should be", p[0], "- yours is", m[0])
            return False
        if m != p:
            print("❌ layer", i + 1, "(" + p[0] + ") should be", p[1:], "- yours is", m[1:])
            return False
    count = sum(p.numel() for p in model.parameters())
    if count != n_params:
        print("❌ the model should have", n_params, "parameters, yours has", count)
        return False
    print("✅ the model matches the plan -", count, "parameters")
    return True


def check_model(model):
    plan = [("Linear", 784, 256), ("ReLU",), ("Dropout", 0.2), ("Linear", 256, 128), ("ReLU",), ("Linear", 128, 10)]
    _compare(model, plan, 235146)


def check_training(val_acc):
    if val_acc is None:
        print("❌ train_model returned nothing - does it end with `return val_acc`?")
    elif not 0 <= val_acc <= 1:
        print("❌ the accuracy should be a share between 0 and 1 - yours is", val_acc)
    elif val_acc < 0.80:
        print("❌ validation accuracy", round(val_acc, 3), "- should be above 0.80 with LR = 0.001 and 3 epochs."
              " Did you call opt.step()? Did you use model.eval() and the VALIDATION images?")
    else:
        print("✅ validation accuracy", round(val_acc, 3))


def check_make_model(make_model):
    for hidden in [32, 512]:
        model = make_model(hidden)
        plan = [("Linear", 784, hidden), ("ReLU",), ("Dropout", 0.2), ("Linear", hidden, 128), ("ReLU",),
                ("Linear", 128, 10)]
        n = 784 * hidden + hidden + hidden * 128 + 128 + 128 * 10 + 10
        print("make_model(" + str(hidden) + "):", end=" ")
        if not _compare(model, plan, n):
            return


def show_results(results, columns):
    """Print a list of dicts as a small table."""
    if len(results) == 0:
        print("no results yet")
        return
    print("   ".join(f"{c:>11}" for c in columns))
    for r in results:
        print("   ".join(f"{round(r[c], 4):>11}" for c in columns))


def check_grid(results, best):
    if len(results) != 9:
        print("❌ results should have 9 entries (3 learning rates × 3 sizes) - it has", len(results))
    elif best is None:
        print("❌ best is still None - pick the entry with the highest validation accuracy")
    elif best["val_acc"] != max(r["val_acc"] for r in results):
        print("❌ best has val_acc", round(best["val_acc"], 4), "- but the highest in results is",
              round(max(r["val_acc"] for r in results), 4))
    else:
        print("✅ best setting: lr =", best["lr"], "  hidden =", best["hidden"], "  val_acc =", round(best["val_acc"], 4))


def test_accuracy(model):
    model.eval()
    with torch.no_grad():
        return round((model(X_test).argmax(1) == y_test).float().mean().item(), 4)


# ---------- Part 3: EEG ----------
def split_by_recording(F, y, rec, val_rec=0, test_rec=2):
    """Train = 4 recordings, validation = 1 recording, test = 1 recording. Standardised with the training part."""
    tr, va, te = (rec != val_rec) & (rec != test_rec), rec == val_rec, rec == test_rec
    mu, sd = F[tr].mean(0), F[tr].std(0) + 1e-6
    as_t = lambda m: torch.tensor((F[m] - mu) / sd)
    return as_t(tr), torch.tensor(y[tr]).long(), as_t(va), y[va], as_t(te), y[te]


def train_eeg(model, weight_seizure=1.0, lr=0.01, data="val"):
    """Train an EEG model (full batch, 200 steps, weighted loss).
    Returns (scores, pred) on the validation (or test) recording: seizure score 0...1 and 0/1 decision at 0.5."""
    if model is None:
        print("❌ no EEG model yet - finish make_eeg_model() in 3.1")
        return None, None
    torch.manual_seed(0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, weight_seizure]))
    model.train()
    for step in range(200):
        loss = loss_fn(model(F_train), y_train_eeg)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        scores = model(F_val if data == "val" else F_test).softmax(1)[:, 1].numpy()
    return scores, (scores >= 0.5).astype(int)


def check_eeg_model(make_eeg_model):
    plan = [("Linear", 90, 32), ("ReLU",), ("Dropout", 0.3), ("Linear", 32, 2)]
    _compare(make_eeg_model(), plan, 90 * 32 + 32 + 32 * 2 + 2)


def false_alarms_per_hour(y, pred, window_s=4.0):
    """Flagged background windows in a row = ONE alarm. Divided by the hours of background (one recording)."""
    fp = (pred == 1) & (y == 0)
    alarms = int(fp[0]) + int(np.sum(fp[1:] & ~fp[:-1]))      # count the starts of runs
    return round(alarms / ((y == 0).sum() * window_s / 3600), 1)


def check_metrics(acc, sens, spec, prec, auprc, scores, pred):
    if pred is None:
        print("❌ no predictions yet - finish 3.1 first")
        return
    true = {"accuracy": accuracy_score(y_val_eeg, pred), "sensitivity": recall_score(y_val_eeg, pred),
            "specificity": recall_score(y_val_eeg, pred, pos_label=0),
            "precision": precision_score(y_val_eeg, pred, zero_division=0),
            "AUPRC": average_precision_score(y_val_eeg, scores)}
    yours = {"accuracy": acc, "sensitivity": sens, "specificity": spec, "precision": prec, "AUPRC": auprc}
    for name in true:
        if yours[name] is None:
            print("❌", name, "- not computed yet")
        elif abs(yours[name] - true[name]) > 1e-6:
            print("❌", name, "=", round(yours[name], 3), "- should be", round(true[name], 3))
        else:
            print("✅", name, "=", round(yours[name], 3))
    print("   false alarms per hour (given):", false_alarms_per_hour(y_val_eeg, pred))


def eeg_grid_search(weights, lrs):
    """Train one EEG model per setting; return a list of dicts with the validation metrics."""
    results = []
    for weight in weights:
        for lr in lrs:
            scores, pred = train_eeg(make_eeg_model(), weight_seizure=weight, lr=lr)
            if scores is None:
                return []
            results.append({"weight": weight, "lr": lr,
                            "accuracy": accuracy_score(y_val_eeg, pred),
                            "sensitivity": recall_score(y_val_eeg, pred),
                            "specificity": recall_score(y_val_eeg, pred, pos_label=0),
                            "auprc": average_precision_score(y_val_eeg, scores),
                            "false_alarms_h": false_alarms_per_hour(y_val_eeg, pred)})
    return results


def check_selection(results, best_by_accuracy, best_by_auprc):
    if len(results) == 0:
        print("❌ no grid results yet")
        return
    for name, best, key in [("best_by_accuracy", best_by_accuracy, "accuracy"), ("best_by_auprc", best_by_auprc, "auprc")]:
        if best is None:
            print("❌", name, "is still None")
        elif best[key] != max(r[key] for r in results):
            print("❌", name, "does not have the highest", key)
        else:
            print("✅", name + ": weight", best["weight"], " lr", best["lr"], "->  sensitivity",
                  round(best["sensitivity"], 2), "  false alarms/h", best["false_alarms_h"], "  AUPRC", round(best["auprc"], 3))


def show_interval(values):
    """The middle 95 % of a list of bootstrap results."""
    if len(values) == 0:
        print("no results yet - fill in the missing line")
        return
    low, high = torch.quantile(torch.tensor(values), torch.tensor([0.025, 0.975])).tolist()
    print("95 % interval:", round(low, 3), "to", round(high, 3))


print("helpers ready")

# Part 1 - Build it yourself (about 30 min)
## 1.1 A model from a plan
Build this network with `nn.Sequential(...)`. The layers come in exactly this order:

| # | layer | in | out | note |
|---|---|---|---|---|
| 1 | `nn.Linear` | 784 | 256 | 784 = one pixel value per input |
| 2 | `nn.ReLU` | | | |
| 3 | `nn.Dropout` | | | switch off 20 % of the neurons (`p=0.2`) |
| 4 | `nn.Linear` | 256 | 128 | |
| 5 | `nn.ReLU` | | | |
| 6 | `nn.Linear` | 128 | 10 | one raw score (logit) per kind of clothes - no softmax! |

In [ ]:
torch.manual_seed(0)    # same start weights for everybody
model = None
# 1. model = nn.Sequential( ... the 6 layers from the plan, separated by commas ... )
pass  # YOUR CODE HERE
print(model)

<details><summary>💡 Hint</summary>

- a layer with 784 inputs and 256 outputs: `nn.Linear(784, 256)`
- activation: `nn.ReLU()` - dropout: `nn.Dropout(p=0.2)` - all with `()`
- `nn.Sequential(layer1, layer2, ...)` puts them in a row
</details>

In [ ]:
check_model(model)

❓ Where do the 235 146 parameters come from? Compute the first layer by hand: 784 × 256 weights + 256 biases.

*Your answer:*

<details><summary>👀 What you should see</summary>

- ✅ the model matches the plan - **235146** parameters
- first layer: 784 × 256 + 256 = **200 960**, second: 256 × 128 + 128 = **32 896**, third: 128 × 10 + 10 = **1 290**
- ReLU and dropout have no parameters
</details>

## 1.2 The training pipeline
Now write the function `train_model(model, lr, epochs)`. It trains the model on the training images and returns
the **validation accuracy**. You know every line from the slide "The training loop: seven knobs" and code break 1.

| step | what | PyTorch |
|---|---|---|
| 1 | optimiser | `torch.optim.Adam(model.parameters(), lr=lr)` |
| 2 | repeat for every epoch | `for epoch in range(epochs):` + `model.train()` |
| 3 | repeat for every batch | `for x, y in loader:` |
| 4 | the 5 training lines | forward → loss → `zero_grad()` → `backward()` → `step()` |
| 5 | validation | `model.eval()`, `with torch.no_grad():`, predictions for `X_val` |
| 6 | accuracy | share of correct predictions, a number between 0 and 1 |

In [ ]:
def train_model(model, lr, epochs):
    """Train `model` on the training images. Return the validation accuracy (0...1)."""
    torch.manual_seed(0)                                   # same batch order for every run
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    val_acc = None
    # 1. create the optimiser (Adam, learning rate lr)
    # 2. for every epoch: switch the model to training mode
    # 3.     for every batch (x, y) in the loader:
    # 4.         forward, loss, zero_grad, backward, step
    #        (optional: print the epoch and the loss of the last batch)
    # 5. switch to eval mode; without gradients: predictions = model(X_val).argmax(1)
    # 6. val_acc = share of predictions that are equal to y_val  (as a Python number: .item())
    pass  # YOUR CODE HERE
    return val_acc

<details><summary>💡 Hint</summary>

- steps 3-4 are exactly the loop from code break 1:
  `loss = loss_fn(model(x), y)`, `opt.zero_grad()`, `loss.backward()`, `opt.step()`
- step 5: `model.eval()` and then indented below `with torch.no_grad():` the line
  `predictions = model(X_val).argmax(1)` (`argmax(1)` = the class with the highest score)
- step 6: `(predictions == y_val)` gives True/False for every image; `.float().mean()` makes the share of
  True; `.item()` turns it into a normal number
</details>

In [ ]:
LR = 0.001      # 👈
EPOCHS = 3      # 👈

import copy
fresh_model = copy.deepcopy(model)      # a fresh copy of your untrained model from 1.1 - so every run starts the same
val_acc = train_model(fresh_model, lr=LR, epochs=EPOCHS)
check_training(val_acc)

🔧 **Change it:** `LR = 0.1` - what happens? Then `EPOCHS = 10` (with `LR = 0.001`).

<details><summary>👀 What you should see</summary>

- LR 0.001, 3 epochs: last batch loss **0.52 → 0.45 → 0.33**, ✅ validation accuracy **0.836**
- LR 0.1: the loss stays high (**1.7-2.2**), ❌ **0.204** - the steps are far too large for Adam (block 1)
- 10 epochs: **0.859** - a bit better, but 3× the time. The last batch loss jumps (0.17 ... 0.56): one batch of
  64 images is a noisy measurement
</details>

# Part 2 - Grid search (about 20 min)
Which learning rate and which size of the first hidden layer are best? Instead of trying by hand, a
**grid search** trains one model for *every* combination and keeps the result.

## 2.1 A model "factory"
For the grid search we need a new model for every combination. Write `make_model(hidden)`: the model from 1.1,
but the first hidden layer has `hidden` neurons instead of 256.

| # | layer | in | out |
|---|---|---|---|
| 1 | `nn.Linear` | 784 | **hidden** |
| 2-3 | `nn.ReLU`, `nn.Dropout(p=0.2)` | | |
| 4 | `nn.Linear` | **hidden** | 128 |
| 5-6 | `nn.ReLU`, `nn.Linear` | 128 | 10 |

In [ ]:
def make_model(hidden):
    """The model from 1.1 with `hidden` neurons in the first hidden layer."""
    torch.manual_seed(0)
    model = None
    # 1. copy your nn.Sequential from 1.1 and replace 256 by hidden (twice!)
    pass  # YOUR CODE HERE
    return model


check_make_model(make_model)

## 2.2 The grid search
Two loops, one inside the other: for every learning rate, for every size → build, train, remember.
9 models × 3 epochs take about 1 minute.

In [ ]:
LRS = [0.0001, 0.001, 0.01]     # 👈 learning rates to try
HIDDENS = [32, 128, 512]        # 👈 sizes of the first hidden layer
EPOCHS = 3

results = []
# 1. for lr in LRS:
# 2.     for hidden in HIDDENS:
# 3.         model = make_model(hidden)
# 4.         val_acc = train_model(model, lr, EPOCHS)
# 5.         results.append({"lr": lr, "hidden": hidden, "val_acc": val_acc})
pass  # YOUR CODE HERE
show_results(results, ["lr", "hidden", "val_acc"])

Now pick the best entry of `results` - the one with the **highest validation accuracy**.

In [ ]:
best = None # dict with best "hidden", "lr", "val_acc"
# 1. start with the first entry: best = results[0]
# 2. go through all entries r in results: if r["val_acc"] is larger than best["val_acc"], then best = r
pass  # YOUR CODE HERE
check_grid(results, best)

<details><summary>💡 Hint</summary>

- a dictionary entry: `r["val_acc"]`
- the loop: `for r in results:` and indented below it `if r["val_acc"] > best["val_acc"]:` and indented again `best = r`
- shorter (for experts): `best = max(results, key=lambda r: r["val_acc"])`
</details>

Only now - **once** - the test images: train the best setting again and measure the test accuracy.

In [ ]:
if best is not None:
    final_model = make_model(best["hidden"])
    train_model(final_model, best["lr"], EPOCHS)
    print("validation accuracy:", round(best["val_acc"], 4), "  test accuracy:", test_accuracy(final_model))

❓ **Explain in one sentence:** why do we choose the best setting with the *validation* images and not with the
test images?

*Your answer:*

<details><summary>👀 What you should see</summary>

- the table: lr 0.0001 gives **0.75-0.82** (too slow in 3 epochs), lr 0.001 **0.83-0.85**, lr 0.01 **0.79-0.83**
  (a bit too large); more neurons help with the small learning rates
- ✅ best setting: lr **0.001**, hidden **512**, val_acc **0.851**
- test accuracy **0.838** - close to, but a little below the validation accuracy (the choice was made on validation)
- answer: if we choose with the test images, the test images help with the choice - then the test result is too
  optimistic (leakage, slide "Data leakage"). The test set must stay untouched until the very end.
</details>

# Part 3 - The right metric in the right place (about 25 min)
FashionMNIST has 10 equally common classes - there the accuracy is fine. Seizures are **rare**. Is accuracy still
a good choice?

Data: patient chb01, 6 one-hour recordings with one seizure each, 4-s windows, 90 band-power numbers per window
(5 frequency bands × 18 channels). Split **by recording**: 4 for training, 1 for validation, 1 for test.
The next cells install the EEG reader and load the data. **The download takes about 1-2 minutes.**

In [ ]:
!pip install -q mne

In [ ]:
"""Small loaders for the two open-access EEG seizure datasets used in the labs.

CHB-MIT  (Shoeb 2009, PhysioNet): paediatric scalp EEG, 23 bipolar channels, 256 Hz, 1 h records.
SeizeIT2 (Bhagubai et al. 2025, OpenNeuro ds005873): wearable behind-the-ear EEG, 2 channels, 256 Hz.
Files are downloaded once and cached in ./data (in Colab: lost when the runtime is recycled).
"""
import re
import urllib.request
from pathlib import Path

import numpy as np

DATA = Path("data")
CHBMIT = "https://physionet-open.s3.amazonaws.com/chbmit/1.0.0/"  # AWS mirror of PhysioNet, much faster
SEIZEIT2 = "https://s3.amazonaws.com/openneuro.org/ds005873/"
# the first 18 CHB-MIT channels form the longitudinal bipolar "double banana" montage
CHANNELS = ["FP1-F7", "F7-T7", "T7-P7", "P7-O1", "FP1-F3", "F3-C3", "C3-P3", "P3-O1", "FP2-F4",
            "F4-C4", "C4-P4", "P4-O2", "FP2-F8", "F8-T8", "T8-P8", "P8-O2", "FZ-CZ", "CZ-PZ"]


def fetch(url, name):
    """Download url to data/name once; return the local path."""
    out = DATA / name
    if not out.exists():
        out.parent.mkdir(parents=True, exist_ok=True)
        print("downloading", url)
        urllib.request.urlretrieve(url, out.with_suffix(".part"))
        out.with_suffix(".part").rename(out)  # no half-written files in the cache
    return out


def preprocess(raw, picks, fs):
    """Band-pass 0.5-40 Hz, resample to fs, return (channels, time) in microvolts."""
    raw.pick(picks).filter(0.5, 40, verbose="error").resample(fs, verbose="error")
    return raw.get_data() * 1e6


def windows(data, fs, win_s, events, around_s=None):
    """Cut (channels, time) into non-overlapping windows of win_s seconds.

    y = 1 if the window centre lies inside a seizure event (onset_s, end_s).
    around_s: keep only windows within around_s seconds of a seizure (None = keep all).
    """
    n = int(win_s * fs)
    k = data.shape[1] // n
    X = data[:, :k * n].reshape(data.shape[0], k, n).transpose(1, 0, 2)
    mid = (np.arange(k) + 0.5) * win_s
    y = np.zeros(k, dtype=np.int64)
    keep = np.zeros(k, dtype=bool) if around_s is not None else np.ones(k, dtype=bool)
    for a, b in events:
        y[(mid >= a) & (mid < b)] = 1
        if around_s is not None:
            keep |= (mid >= a - around_s) & (mid < b + around_s)
    return np.ascontiguousarray(X[keep], dtype=np.float32), y[keep]


def chbmit_seizures(patient="chb01"):
    """{'chb01_03': [(2996, 3036)], ...} parsed from the PhysioNet summary file."""
    text = fetch(f"{CHBMIT}{patient}/{patient}-summary.txt", f"chbmit/{patient}-summary.txt").read_text()
    out = {}
    for block in text.split("File Name: ")[1:]:
        t = [int(s) for s in re.findall(r"Seizure.*(?:Start|End) Time:\s*(\d+)", block)]
        out[block.split(".edf")[0]] = list(zip(t[::2], t[1::2]))
    return out


def load_chbmit(records=("chb01_03", "chb01_04", "chb01_15", "chb01_16", "chb01_18", "chb01_26"),
                win_s=4.0, fs=128):
    """CHB-MIT records -> X (n, 18, win_s*fs) in uV, y (n,) 1 = seizure, rec (n,) index into records."""
    import mne
    X, y, rec = [], [], []
    for i, r in enumerate(records):
        patient = r.split("_")[0]
        raw = mne.io.read_raw_edf(fetch(f"{CHBMIT}{patient}/{r}.edf", f"chbmit/{r}.edf"), preload=True, verbose="error")
        raw.rename_channels(lambda c: c.removesuffix("-0"))  # MNE renames the duplicated T8-P8 channel
        Xi, yi = windows(preprocess(raw, CHANNELS, fs), fs, win_s, chbmit_seizures(patient)[r])
        X.append(Xi), y.append(yi), rec.append(np.full(len(yi), i))
    return np.concatenate(X), np.concatenate(y), np.concatenate(rec)


def seizeit2_events(subject="sub-001"):
    """{run_path: [(onset_s, end_s), ...]} seizure events of every EEG run of one SeizeIT2 subject."""
    listing = urllib.request.urlopen(
        f"https://s3.amazonaws.com/openneuro.org?list-type=2&prefix=ds005873/{subject}/").read().decode()
    out = {}
    for stem in re.findall(r"<Key>ds005873/([^<]+?/eeg/[^<]+?)_events\.tsv</Key>", listing):
        rows = fetch(f"{SEIZEIT2}{stem}_events.tsv", f"seizeit2/{stem}_events.tsv").read_text().splitlines()[1:]
        cols = [r.split("\t") for r in rows]
        out[stem] = [(float(c[0]), float(c[0]) + float(c[1])) for c in cols if c[2].startswith("sz")]
    return out


def load_seizeit2(subject="sub-001", win_s=4.0, fs=128, around_s=1800):
    """Seizure-containing runs of one SeizeIT2 subject -> X (n, 2, win_s*fs) in uV, y (n,), run (n,).

    around_s keeps +-30 min around each seizure, otherwise one subject is >100 h of mostly background.
    """
    import mne
    X, y, run = [], [], []
    for i, (stem, events) in enumerate(e for e in seizeit2_events(subject).items() if e[1]):
        raw = mne.io.read_raw_edf(fetch(f"{SEIZEIT2}{stem}_eeg.edf", f"seizeit2/{stem}_eeg.edf"), preload=True, verbose="error")
        Xi, yi = windows(preprocess(raw, raw.ch_names[:2], fs), fs, win_s, events, around_s)
        X.append(Xi), y.append(yi), run.append(np.full(len(yi), i))
    return np.concatenate(X), np.concatenate(y), np.concatenate(run)


def band_power(X, fs=128):
    """Log band power (delta, theta, alpha, beta, low gamma) per channel: (n, ch, t) -> (n, ch*5)."""
    from scipy.signal import welch
    f, p = welch(X, fs=fs, nperseg=fs, axis=-1)
    bands = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 40)]
    bp = np.stack([p[..., (f >= a) & (f < b)].mean(-1) for a, b in bands], -1)
    return np.log(bp + 1e-10).reshape(len(X), -1).astype(np.float32)

In [ ]:
X_eeg, y_eeg, rec = load_chbmit()           # windows, labels (1 = seizure), recording number 0-5
F_eeg = band_power(X_eeg)                   # 90 band-power features per window
F_train, y_train_eeg, F_val, y_val_eeg, F_test, y_test_eeg = split_by_recording(F_eeg, y_eeg, rec)
print("training windows:", len(y_train_eeg), "  seizure windows among them:", int(y_train_eeg.sum()))
print("validation windows:", len(y_val_eeg), "  seizure:", int(y_val_eeg.sum()),
      "  test windows:", len(y_test_eeg), "  seizure:", int(y_test_eeg.sum()))

## 3.1 An EEG model from a plan
Write `make_eeg_model()`. Training is done by the given helper `train_eeg(...)`.

| # | layer | in | out | note |
|---|---|---|---|---|
| 1 | `nn.Linear` | 90 | 32 | 90 band-power numbers |
| 2 | `nn.ReLU` | | | |
| 3 | `nn.Dropout` | | | `p=0.3` |
| 4 | `nn.Linear` | 32 | 2 | two classes: background, seizure |

In [ ]:
def make_eeg_model():
    torch.manual_seed(0)    # same start weights for everybody
    model = None
    # 1. model = nn.Sequential( ... the 4 layers from the plan ... )
    pass  # YOUR CODE HERE
    return model


check_eeg_model(make_eeg_model)

## 3.2 Compute the metrics yourself
`train_eeg` returns two things for the validation recording: `scores` (seizure score 0...1 per window) and `pred`
(0 or 1, threshold 0.5). Compute five metrics with scikit-learn. The true labels are `y_val_eeg`.

| metric | scikit-learn | uses |
|---|---|---|
| accuracy | `accuracy_score(y_true, y_pred)` | `pred` |
| sensitivity | `recall_score(y_true, y_pred)` | `pred` |
| specificity | `recall_score(y_true, y_pred, pos_label=0)` | `pred` |
| precision | `precision_score(y_true, y_pred)` | `pred` |
| AUPRC | `average_precision_score(y_true, scores)` | `scores` (no threshold!) |

In [ ]:
WEIGHT_SEIZURE = 1.0    # 👈 later: 58

scores, pred = train_eeg(make_eeg_model(), weight_seizure=WEIGHT_SEIZURE)
acc, sens, spec, prec, auprc = None, None, None, None, None
# 1.-5. one line per metric, e.g.  acc = accuracy_score(y_val_eeg, pred)
pass  # YOUR CODE HERE
check_metrics(acc, sens, spec, prec, auprc, scores, pred)

🔧 **Change it:** `WEIGHT_SEIZURE = 58` - one seizure window counts like 58 background windows.

❓ **Explain in one sentence:** which of the numbers tell you something useful here - and which one hardly moves?

*Your answer:*

<details><summary>👀 What you should see</summary>

- weight 1: accuracy **0.997**, sensitivity **0.70**, specificity **1.0**, precision **1.0**, AUPRC **0.854**,
  **0** false alarms per hour - but 3 of 10 seizure windows missed
- weight 58: accuracy **0.991**, sensitivity **0.90**, specificity **0.992**, precision **0.56**, AUPRC **0.871**,
  **3.0** false alarms per hour
- accuracy and specificity hardly move: 890 of 900 windows are background, so they are almost always ~99 %
- useful: sensitivity (found seizures), precision or false alarms per hour (how often the alarm is wrong),
  AUPRC (all thresholds at once)
- only 10 seizure windows in the validation recording - one more found window changes the sensitivity by 10 %
</details>

## 3.3 Grid search: which metric chooses the model?
The helper `eeg_grid_search` trains 6 models (3 class weights × 2 learning rates) and returns their validation
metrics. Choose the best model **twice**: once by accuracy, once by AUPRC.

In [ ]:
WEIGHTS = [1, 10, 58]       # 👈
LRS_EEG = [0.001, 0.01]     # 👈

eeg_results = eeg_grid_search(WEIGHTS, LRS_EEG)
show_results(eeg_results, ["weight", "lr", "accuracy", "sensitivity", "specificity", "auprc", "false_alarms_h"])

best_by_accuracy = None
best_by_auprc = None
# 1. best_by_accuracy = the entry with the highest "accuracy"
# 2. best_by_auprc    = the entry with the highest "auprc"
pass  # YOUR CODE HERE
check_selection(eeg_results, best_by_accuracy, best_by_auprc)

<details><summary>💡 Hint</summary>

- the same loop as in 2.2 (`best = eeg_results[0]`, then `for r in eeg_results: ...`) - once with `"accuracy"`,
  once with `"auprc"`
</details>

❓ **Explain in one sentence:** which of the two chosen models would you give to a doctor - and why?

*Your answer:*

<details><summary>👀 What you should see</summary>

- accuracy picks weight **1**, lr **0.01**: sensitivity **0.70** (3 of 10 seizure windows missed), 0 false alarms/h
- AUPRC picks weight **58**, lr **0.001**: sensitivity **0.90**, but **12.1** false alarms/h at threshold 0.5;
  AUPRC **0.887** (vs 0.854)
- all 6 accuracies are between **0.970** and **0.997** - accuracy prefers the model that says "seizure" least often
- answer: neither as it is - take the AUPRC model (it ranks seizure windows best at every threshold) and then
  choose the threshold on validation data so that the false alarms are acceptable (3.4, slide "Choosing the
  decision threshold"). Accuracy would never lead you there.
</details>

## 3.4 Which metric where?
Fill in the right column (double-click this cell). Use the slides of block 3.

| situation | which metric(s)? |
|---|---|
| choose the best FashionMNIST model (10 equally common classes) | |
| choose the best seizure model in a grid search | |
| choose the threshold of the seizure detector | |
| tell a doctor how good the detector is | |
| the results table of your project | |

<details><summary>👀 What you should see</summary>

- FashionMNIST: **accuracy** on the validation set is fine - the classes are balanced
- seizure model selection: **AUPRC** (or validation loss) on the validation set - no threshold needed, not fooled by
  the many background windows
- threshold: **false alarms per hour** together with **sensitivity**, on validation recordings - the doctor's
  limit (e.g. 1 per day) decides
- for the doctor: **sensitivity** (events: found seizures) and **false alarms per hour** on the *test* recordings,
  at the fixed threshold
- your project: one **main metric chosen in advance** + sensitivity / specificity (or precision) / F1 / AUPRC,
  mean ± std over seeds and a 95 % interval - never accuracy alone
</details>

## ⭐ Bonus: how sure are we about the test AUPRC?
The model chosen by AUPRC is trained again and scored **once** on the test recording. Then the bootstrap from the
slides: draw the test windows 2000 times with replacement, compute the AUPRC each time.

✏️ **Fill in one line:** add the AUPRC of this draw to the list `draws` (use `y_test_eeg[idx]` and `scores_test[idx]`).

In [ ]:
draws = []
if best_by_auprc is not None:
    scores_test, pred_test = train_eeg(make_eeg_model(), weight_seizure=best_by_auprc["weight"],
                                       lr=best_by_auprc["lr"], data="test")
    print("test AUPRC:", round(average_precision_score(y_test_eeg, scores_test), 3),
          "  sensitivity:", round(recall_score(y_test_eeg, pred_test), 2),
          "  false alarms per hour:", false_alarms_per_hour(y_test_eeg, pred_test))
    torch.manual_seed(0)
    for i in range(2000):
        idx = torch.randint(0, len(y_test_eeg), (len(y_test_eeg),))  # draw test windows with replacement
        pass  # YOUR CODE HERE
show_interval(draws)

<details><summary>👀 What you should see</summary>

- test AUPRC **0.315**, sensitivity **0.80**, **2.0** false alarms per hour on the test recording
- the test AUPRC is much lower than the validation AUPRC (0.887): this seizure looks different, and one validation
  recording with 10 seizure windows is a small basis for choosing
- 95 % interval **0.16 to 0.586** - wide: only 10 seizure windows in the test recording
- careful: all windows come from one patient and neighbours are similar - the real uncertainty is even larger
</details>

# For your project
What you wrote today is the skeleton of every project:
- a **model** from a plan (`nn.Sequential`) and a **training function**
- a **grid search** on the *validation* data - the test data is used **once**, at the end
- **split** so that the test data looks like the real situation (new recording, new patient)
- the **right metric** for each step: AUPRC / validation loss for choosing, false alarms per hour for the threshold,
  sensitivity *and* specificity (or precision) for reporting - never accuracy alone on imbalanced data
- **several seeds** (mean ± std) and a **95 % interval**